# Work 1: Source-Separated Track/Distance Form
**Goal:** Calculate smoothed historical win, top-3, and finishing-percentile rates by source, track, distance band, and track-distance combination. <br>
**Method:** Use strictly prior race history (`shift(1)`), apply minimum-history shrinkage toward the global rate, and convert to within-race relative measures.<br>
**Input:** `runners_merged_with_features.parquet`<br>
**Output:** `runners_merged_work1.parquet`

In [1]:
import pandas as pd
import numpy as np

# Load the dataset from Week 3
df = pd.read_parquet('runners_merged_with_features.parquet')

# Filter for valid races and starters
m = df[df['race_valid'] & df['started']].copy()

# Sort chronologically to ensure historical calculations are correct
m['race_date'] = pd.to_datetime(m['race_date'])
m = m.sort_values(by=['source', 'race_date', 'race_no']).reset_index(drop=True)

# Define Distance Bands (Sprint, Mile, Middle, Stay)
def get_distance_band(d):
    if d <= 1200: return 'sprint'
    elif d <= 1650: return 'mile'
    elif d <= 1800: return 'middle'
    else: return 'stay'

m['distance_band'] = m['distance_m'].apply(get_distance_band)
m['track_dist_combo'] = m['track'] + '_' + m['distance_band']

# Calculate finishing percentile (1.0 = win, 0.0 = last)
# Handle field_size == 1 edge case to avoid division by zero
m['finish_pct'] = 1 - ((m['finish_pos'] - 1) / (m['field_size'] - 1).replace(0, 1))

print(f"Data loaded. Total rows: {len(m)}")

Data loaded. Total rows: 38942


### Step 1: Define Shrinkage Function
To prevent overfitting on horses with very few starts at a specific track/distance, we apply Bayesian shrinkage (k=5). This pulls the historical rate toward the global source average if the sample size is small.

In [5]:
# Calculate source-specific global averages (to shrink towards)
source_global_win = m.groupby('source')['is_winner'].transform('mean')
source_global_top3 = m.groupby('source')['is_top3'].transform('mean')
source_global_pct = m.groupby('source')['finish_pct'].transform('mean')

def calculate_smoothed_features(data, group_cols, prefix):
    # Shift targets by 1 to exclude current race
    win_s = data.groupby(group_cols)['is_winner'].shift(1)
    top3_s = data.groupby(group_cols)['is_top3'].shift(1)
    pct_s = data.groupby(group_cols)['finish_pct'].shift(1)

    # Assign to temp columns to avoid index mismatch during transform
    data['_tmp_win'] = win_s
    data['_tmp_top3'] = top3_s
    data['_tmp_pct'] = pct_s

    # Calculate expanding counts and sums safely using transform
    runs = data.groupby(group_cols)['_tmp_win'].transform(lambda x: x.expanding().count())
    wins = data.groupby(group_cols)['_tmp_win'].transform(lambda x: x.expanding().sum())
    top3s = data.groupby(group_cols)['_tmp_top3'].transform(lambda x: x.expanding().sum())
    pcts = data.groupby(group_cols)['_tmp_pct'].transform(lambda x: x.expanding().sum())

    # Fill NaNs with 0 for calculation
    runs = runs.fillna(0); wins = wins.fillna(0); top3s = top3s.fillna(0); pcts = pcts.fillna(0)

    k = 5 # Shrinkage parameter

    # Smoothed formulas
    data[f'{prefix}_win'] = (wins + k * source_global_win) / (runs + k)
    data[f'{prefix}_top3'] = (top3s + k * source_global_top3) / (runs + k)
    data[f'{prefix}_pct'] = (pcts + k * source_global_pct) / (runs + k)

    # Drop the temporary columns to keep the dataset clean
    data = data.drop(columns=['_tmp_win', '_tmp_top3', '_tmp_pct'])

    return data

### Step 2: Calculate Track, Distance, and Combo Forms
We calculate three separate sets of historical rates:
1. **Track Form:** e.g., How good is the horse specifically at Sha Tin?
2. **Distance Form:** e.g., How good is the horse specifically in Sprint races?
3. **Track-Distance Combo:** e.g., How good is the horse at Sha Tin in Sprint races?

In [6]:
# 1. Track Form
m = calculate_smoothed_features(m, ['source', 'horse_id', 'track'], 'track')

# 2. Distance Band Form
m = calculate_smoothed_features(m, ['source', 'horse_id', 'distance_band'], 'dist')

# 3. Track-Distance Combo Form
m = calculate_smoothed_features(m, ['source', 'horse_id', 'track_dist_combo'], 'combo')

print("Historical forms calculated with shrinkage.")

Historical forms calculated with shrinkage.


### Step 3: Convert to Within-Race Relative Measures
A horse's absolute win rate doesn't tell us much without comparing it to its rivals in the *same* race. We subtract the race average from each feature to create relative measures.

In [7]:
features_to_rel = [
    'track_win', 'track_top3', 'track_pct',
    'dist_win', 'dist_top3', 'dist_pct',
    'combo_win', 'combo_top3', 'combo_pct'
]

for f in features_to_rel:
    m[f'rel_{f}'] = m[f] - m.groupby('race_id')[f].transform('mean')

print("Within-race relative measures created.")

Within-race relative measures created.


### Step 4: Correlation Check & Save
Let's quickly check the correlation of our new relative features with winning, then save the final dataset for Fahim.

In [9]:
rel_features = [col for col in m.columns if col.startswith('rel_')]
correlation = m[rel_features + ['is_winner']].corr()['is_winner'].sort_values(ascending=False)
print("Correlation with winning (is_winner):")
print(correlation)

# Save the final dataset with Week 3 + Work 1 features
m.to_parquet('version2_runners_merged.parquet')
print("\nDataset successfully saved as 'version2_runners_merged.parquet'")

Correlation with winning (is_winner):
is_winner         1.000000
rel_track_pct     0.142695
rel_track_top3    0.135887
rel_dist_pct      0.135880
rel_combo_pct     0.133115
rel_dist_top3     0.125899
rel_combo_top3    0.123890
rel_track_win     0.106516
rel_combo_win     0.101932
rel_dist_win      0.101340
Name: is_winner, dtype: float64

Dataset successfully saved as 'version2_runners_merged.parquet'
